In [1]:
# [Cell 1] Path Resolution & Environment Setup
import os
import sys
from pathlib import Path

def get_project_root() -> Path:
    """
    Dynamically resolves the project root.
    Assumes the notebook is either in the root or in the 'notebooks/' directory.
    """
    current_path = Path(__file__).parent.resolve() if '__file__' in locals() else Path.cwd()
    
    # Walk up to find pyproject.toml or .git to identify the root
    for parent in [current_path] + list(current_path.parents):
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
            
    # Fallback: assume current working directory is root if running from root
    return current_path

PROJECT_ROOT = get_project_root()
print(f"✅ Project Root Resolved: {PROJECT_ROOT}")

# Ensure src is in the Python path for imports
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    print(f"✅ Added {SRC_PATH} to sys.path")

# Create necessary directories if they don't exist (for local experiment)
(PROJECT_ROOT / "logs").mkdir(exist_ok=True)
(PROJECT_ROOT / "data" / "raw" / "official_laws").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "raw" / "user_uploads").mkdir(parents=True, exist_ok=True)
print("✅ Core directories verified/created.")

✅ Project Root Resolved: /run/media/ahmed/Data/RAG/Egyptian-Legal-AI
✅ Added /run/media/ahmed/Data/RAG/Egyptian-Legal-AI/src to sys.path
✅ Core directories verified/created.


In [2]:
# [Cell 2] Logging Configuration
import logging
import sys
from datetime import datetime

def setup_logger(name: str, level: str = "INFO") -> logging.Logger:
    """Configures and returns a standardized logger."""
    logger = logging.getLogger(name)
    
    # Prevent duplicate handlers if cell is re-run
    if logger.hasHandlers():
        logger.handlers.clear()
        
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    
    # Production-ready format (can be swapped for JSON later)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    
    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    # File handler (optional, validates path resolution)
    log_dir = PROJECT_ROOT / "logs"
    log_dir.mkdir(exist_ok=True)
    file_handler = logging.FileHandler(log_dir / "foundation_experiment.log", encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    
    return logger

# Initialize the foundation logger
logger = setup_logger("egyptian_legal_ai.foundation", level="DEBUG")
logger.info("Logger initialized successfully.")
logger.debug("This is a debug message to verify log levels.")

2026-10-31 11:52:45 | INFO     | egyptian_legal_ai.foundation | Logger initialized successfully.
2026-10-31 11:52:45 | DEBUG    | egyptian_legal_ai.foundation | This is a debug message to verify log levels.


In [3]:
# [Cell 3] Configuration & Environment Variables (Pydantic Settings)
from pydantic_settings import BaseSettings
from pydantic import Field, ConfigDict, ValidationError
from typing import Optional

# Create a mock .env file for the experiment if it doesn't exist
env_path = PROJECT_ROOT / ".env"
if not env_path.exists():
    with open(env_path, "w", encoding="utf-8") as f:
        f.write("# Egyptian Legal AI - Local Experiment Env\n")
        f.write("APP_ENV=development\n")
        f.write("LOG_LEVEL=DEBUG\n")
        f.write("QDRANT_HOST=localhost\n")
        f.write("QDRANT_PORT=6333\n")
    logger.info(f"Created mock .env file at {env_path}")

class Settings(BaseSettings):
    """Centralized application settings."""
    
    model_config = ConfigDict(
        env_file=str(PROJECT_ROOT / ".env"),
        env_file_encoding="utf-8",
        extra="ignore",
        case_sensitive=False
    )
    
    # App Settings
    app_name: str = Field(default="Egyptian Legal AI", description="Application name")
    app_env: str = Field(default="development", description="Environment (development, production)")
    log_level: str = Field(default="INFO", description="Logging level")
    
    # Storage Settings
    qdrant_host: str = Field(default="localhost")
    qdrant_port: int = Field(default=6333)
    postgres_url: Optional[str] = Field(default=None, description="PostgreSQL connection string")
    
    # Paths
    data_dir: Path = Field(default=PROJECT_ROOT / "data")
    artifacts_dir: Path = Field(default=PROJECT_ROOT / "artifacts")

    @property
    def is_development(self) -> bool:
        return self.app_env.lower() == "development"

try:
    settings = Settings()
    logger.info(f"✅ Configuration loaded successfully. Env: {settings.app_env}")
    logger.info(f"   - Qdrant: {settings.qdrant_host}:{settings.qdrant_port}")
    logger.info(f"   - Data Dir: {settings.data_dir}")
except ValidationError as e:
    logger.error(f"❌ Configuration validation failed:\n{e}")
    raise

2026-10-31 11:53:07 | INFO     | egyptian_legal_ai.foundation | ✅ Configuration loaded successfully. Env: development
2026-10-31 11:53:07 | INFO     | egyptian_legal_ai.foundation |    - Qdrant: localhost:6333
2026-10-31 11:53:07 | INFO     | egyptian_legal_ai.foundation |    - Data Dir: data


In [4]:
# [Cell 4] Domain Models & Contracts (Pydantic v2)
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, List
from datetime import datetime
from enum import Enum
import hashlib

class LegalStatus(str, Enum):
    ACTIVE = "active"
    REPEALED = "repealed"
    AMENDED = "amended"
    CONSOLIDATED = "consolidated"

class LegalSource(BaseModel):
    """Represents the provenance and metadata of a legal text."""
    model_config = ConfigDict(str_strip_whitespace=True, use_enum_values=True)
    
    law_name: str = Field(..., description="Official name of the law")
    law_number: str = Field(..., description="Law number or decree number")
    publication_year: int = Field(..., ge=1800, le=2100)
    article_number: Optional[str] = Field(default=None, description="Specific article if applicable")
    chapter: Optional[str] = None
    status: LegalStatus = Field(default=LegalStatus.ACTIVE)
    effective_date: Optional[datetime] = None

class Document(BaseModel):
    """Core document representation."""
    model_config = ConfigDict(str_strip_whitespace=True)
    
    id: str = Field(..., description="Unique document identifier (e.g., UUID or hash)")
    title: str
    source: LegalSource
    content_hash: str = Field(..., description="SHA-256 hash of the raw content")
    file_type: str = Field(..., pattern=r"^(pdf|docx|txt|md|html)$")
    uploaded_at: datetime = Field(default_factory=datetime.utcnow)
    is_official: bool = Field(default=False, description="True if from official gazette, False if user upload")

class Query(BaseModel):
    """User query representation."""
    id: str
    text: str = Field(..., min_length=3, max_length=2000)
    language: str = Field(default="ar", pattern=r"^(ar|en)$")
    created_at: datetime = Field(default_factory=datetime.utcnow)

logger.info("✅ Domain models defined successfully.")

2026-10-31 11:53:22 | INFO     | egyptian_legal_ai.foundation | ✅ Domain models defined successfully.


In [5]:
# [Cell 5] Validation & Error Handling Tests

def test_domain_models():
    logger.info("🧪 Running Domain Model Validation Tests...")
    
    # 1. Test Valid Legal Source & Document
    try:
        source = LegalSource(
            law_name=" القانون المدني ", # Testing strip_whitespace
            law_number="131",
            publication_year=1948,
            status=LegalStatus.ACTIVE
        )
        
        doc = Document(
            id="doc_12345",
            title="Egyptian Civil Code",
            source=source,
            content_hash=hashlib.sha256(b"dummy content").hexdigest(),
            file_type="pdf",
            is_official=True
        )
        assert doc.source.law_name == "القانون المدني" # Ensure whitespace was stripped
        logger.info("✅ Test 1 Passed: Valid Document creation and whitespace stripping.")
    except Exception as e:
        logger.error(f"❌ Test 1 Failed: {e}")
        raise

    # 2. Test Invalid Year (Should fail validation)
    try:
        bad_source = LegalSource(
            law_name="Bad Law",
            law_number="999",
            publication_year=2200 # Invalid: > 2100
        )
        logger.error("❌ Test 2 Failed: Should have raised ValidationError for year 2200.")
        raise AssertionError("Validation failed to catch invalid year.")
    except ValidationError as e:
        logger.info("✅ Test 2 Passed: Correctly rejected invalid publication year.")
        
    # 3. Test Invalid File Type (Should fail regex pattern)
    try:
        bad_doc = Document(
            id="doc_67890",
            title="Bad File",
            source=source,
            content_hash="abc",
            file_type="exe" # Invalid: not in allowed types
        )
        logger.error("❌ Test 3 Failed: Should have raised ValidationError for .exe file.")
        raise AssertionError("Validation failed to catch invalid file type.")
    except ValidationError as e:
        logger.info("✅ Test 3 Passed: Correctly rejected invalid file type.")

    # 4. Test Query Length Constraints
    try:
        short_query = Query(id="q1", text="ما") # Too short (< 3 chars)
        logger.error("❌ Test 4 Failed: Should have raised ValidationError for short query.")
    except ValidationError:
        logger.info("✅ Test 4 Passed: Correctly rejected query shorter than min_length.")

    logger.info("🎉 All foundation validation tests passed successfully!")

# Execute tests
test_domain_models()

2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | 🧪 Running Domain Model Validation Tests...
2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | ✅ Test 1 Passed: Valid Document creation and whitespace stripping.
2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | ✅ Test 2 Passed: Correctly rejected invalid publication year.
2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | ✅ Test 3 Passed: Correctly rejected invalid file type.
2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | ✅ Test 4 Passed: Correctly rejected query shorter than min_length.
2026-10-31 11:53:37 | INFO     | egyptian_legal_ai.foundation | 🎉 All foundation validation tests passed successfully!
